# IK MLP Training

## Config

In [ ]:
SAVE_DATA = "G:/My Drive/inverse_kinematics/hot_start"
SAVE_MODEL = f"{SAVE_DATA}/mlp_ik.pt"

BATCH_SIZE = 512
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 30

## Imports

In [ ]:
import sys
sys.path.insert(0, r"c:\Users\max\InverseKinematics")

import torch
from torch.utils.data import DataLoader

from ik.data.dataset import IKDataset
from ik.model.mlp import MLP
from ik.model.training import run_training, evaluate_and_return_loss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

## Load Data

In [ ]:
train_ds = IKDataset("train", SAVE_DATA)
val_ds   = IKDataset("val",   SAVE_DATA,
                     MinMax_X=train_ds.MinMax_X,
                     MinMax_Y=train_ds.MinMax_Y)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_ds):,} samples")
print(f"val:   {len(val_ds):,} samples")
print(f"input dim: {train_ds.X.shape[1]}  (R6[6] + P[3] + q_init[6])")
print(f"output dim: {train_ds.y.shape[1]}  (q_target[6])")

## Model

In [ ]:
model = MLP(input_dim=15, hidden_dim=256, output_dim=6, n_layers=4).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

## Train

In [ ]:
model = run_training(
    model,
    train_loader,
    val_loader,
    MinMax_Y=train_ds.MinMax_Y,
    device=DEVICE,
    save_path=SAVE_MODEL,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
)

## Evaluate on Val Set

In [ ]:
mse_losses, pos_losses, rot_losses = evaluate_and_return_loss(
    model, val_loader, DEVICE, MinMax_Y=train_ds.MinMax_Y
)